In [ ]:
!pip install torch transformers librosa soundfile torchaudio praat-parselmouth pyannote.audio speechbrain openai-whisper resampy imbalanced-learn

In [ ]:
USE DATABASE AICOLLEGE;
USE SCHEMA PUBLIC;
USE ROLE ACCOUNTADMIN;
SHOW MODELS;


In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry
import pandas as pd

# Get the current Snowflake session
session = get_active_session()

# Initialize the model registry
reg = Registry(session=session, database_name="AICOLLEGE", schema_name="PUBLIC")

# Retrieve the deployed model
model_ref = reg.get_model("NORBERT3_BASE_SENTENCE_SENTIMENT")

# Get the latest version (or specify a version)
model = model_ref.default


In [ ]:
# First, let's see what methods are available
print(dir(model))
print(f"Model type: {type(model)}")

# Check model signature
print(model.show_functions())

In [ ]:
# Sample text data for sentiment analysis
sample_texts = [
    "I love this product! It's amazing.",
    "This is terrible. I hate it.", 
    "The weather is okay today.",
    "Absolutely fantastic experience!",
    "Not sure how I feel about this."
]

# Create a DataFrame with the sample data
df = session.create_dataframe(
    [(i, text) for i, text in enumerate(sample_texts)], 
    schema=["id", "text"]
)

# Use the model's predict method
#predictions = model.predict(df)

# Show results
#predictions.show()

# Use the run method instead of predict
#predictions = model.run(df, function_name="predict")
#predictions.show()

# Receipt

In [ ]:
# Set the external access integration
session = get_active_session()
session.sql("ALTER SESSION SET PYTHON_CONNECTOR_QUERY_RESULT_FORMAT = 'json'").collect()

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry
from transformers import pipeline

# Get session
session = get_active_session()
session.sql("ALTER SESSION SET PYTHON_CONNECTOR_QUERY_RESULT_FORMAT = 'json'").collect()


# Initialize registry
reg = Registry(session=session, database_name="AICOLLEGE", schema_name="PUBLIC")

# Load the Hugging Face model using transformers
hf_model = pipeline(
    "text-classification",
    model="ltg/norbert3-base_sentence-sentiment",
    trust_remote_code=True,
    return_all_scores=True
)

# Log the model to Snowflake Model Registry
model_version = reg.log_model(
    model=hf_model,
    model_name="NORBERT3_SENTIMENT_MODEL", 
    version_name="v1",
    conda_dependencies=["transformers", "torch", "tokenizers"],
    comment="Hugging Face NORBERT3 sentiment analysis model",
    sample_input_data=session.create_dataframe([("Sample text",)], schema=["text"])
)

print(f"Model logged successfully: {model_version}")

In [ ]:
# Deploy to SPCS after logging
model_version.create_service(
    service_name="norbert3-sentiment-service",
    service_compute_pool="MLOPS_COMPUTE_POOL_GPU",  # You need a GPU compute pool
    image_repo="AICOLLEGE.PUBLIC.HFREPO",
    max_instances=1
)

In [ ]:
from transformers import pipeline
import librosa
import warnings
import resampy
from datetime import datetime
import soundfile as sf
from scipy import signal

session = get_active_session()

In [ ]:
emotion_pipeline = pipeline(
            "audio-classification",
            model="ehcalabres/wav2vec2-lg-xlsr-en-speech-emotion-recognition"
)

In [ ]:
emotion_pipeline()